# Семинар 11. Случайный лес

Соберём случайный лес из регрессионных деревьев и проверим его на задаче оценки стоимости жилья. Нас будут интересовать три решения: сколько деревьев обучить, сколько признаков давать им при поиске разбиений и как выбирать настройки по OOB-ошибке.

## 0. Подготовка

Нужны NumPy, pandas, matplotlib и scikit-learn (1.4 или новее). При первой загрузке California Housing понадобится интернет; затем sklearn использует кэш. Для работы без сети см. `DATA_SOURCE.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, ParameterGrid
from sklearn.metrics import root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor

SEED = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.figsize": (8, 4.5)})
pd.set_option("display.precision", 3)

## 1. California Housing: знакомство с задачей

Данные основаны на переписи населения США 1990 года. Одна строка соответствует небольшой статистической территории — группе переписных блоков (*census block group*). Бюро переписи объединяет соседние блоки для публикации статистики: можно представить такую территорию как небольшой район, в котором живут сотни или несколько тысяч человек. Это не административная единица. В таблице записаны агрегированные характеристики территории, а ответ — медианная стоимость жилья на ней. Мы работаем с историческими ценами 1990 года.

`fetch_california_housing(as_frame=True)` возвращает объект с данными и их описанием. Атрибут `.frame` содержит признаки и ответ в одной таблице pandas. sklearn уже рассчитывает среднее число комнат, спален и жильцов на домохозяйство, а стоимость выражает в единицах 100 000 долларов.

| Столбец | Значение |
|---|---|
| `MedInc` | Медианный доход домохозяйств (10 000 долларов) |
| `HouseAge` | Медианный возраст жилья (лет) |
| `AveRooms` | Среднее число комнат на домохозяйство |
| `AveBedrms` | Среднее число спален на домохозяйство |
| `Population` | Численность населения |
| `AveOccup` | Среднее число человек на домохозяйство |
| `Latitude`, `Longitude` | Широта и долгота (градусы) |
| `MedHouseVal` | Медианная стоимость жилья (100 000 долларов) |

Географические единицы: [Census Bureau](https://www.census.gov/glossary/?term=Block+Group+%28BG%29). Источник данных: [StatLib](https://lib.stat.cmu.edu/datasets/houses.zip), R. K. Pace и R. Barry (1997), *Sparse Spatial Autoregressions*; [описание данных в scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset).

In [ ]:
dataset = fetch_california_housing(as_frame=True)
housing = dataset.frame
print("Размер таблицы:", housing.shape)
housing.head()

В таблице 20 640 территорий. Например, медианная стоимость жилья на первой территории — 452,6 тыс. долларов, возраст жилья — 41 год.

**Разбиение.** Сегодня сравним модели на validation. На следующем семинаре добавим бустинг, после чего проверим выбранные модели на test. Отделим 20% данных для test, ещё 20% всей таблицы — для validation. Случайное разбиение оценивает прогноз на территориях из той же совокупности; для проверки переноса в другой регион нужны пространственные разбиения.

In [ ]:
dev, test = train_test_split(housing, test_size=0.2, random_state=SEED)
train, val = train_test_split(dev, test_size=0.25, random_state=SEED + 1)
feature_names = housing.columns.drop("MedHouseVal").tolist()
X_train = train[feature_names].to_numpy()
y_train = train["MedHouseVal"].to_numpy()
X_val = val[feature_names].to_numpy()
y_val = val["MedHouseVal"].to_numpy()
print("Train:", len(train), "| Validation:", len(val), "| Test:", len(test))

In [ ]:
print("Типы признаков:", train[feature_names].dtypes.unique())
print("Всего пропусков:", train.isna().sum().sum())
train.describe().T.round(2)

Пропусков нет, все признаки числовые. Масштабы существенно различаются: население измеряется сотнями и тысячами человек, а число спален — единицами. Для Ridge и kNN используем `StandardScaler` внутри pipeline. Деревья сравнивают значения отдельных признаков с порогами, поэтому масштабирование им не нужно.

Посмотрим, какие зависимости предстоит описать моделям.

In [ ]:
fig, ax = plt.subplots()
ax.hist(y_train, bins=40, color="steelblue", edgecolor="white")
ax.set(xlabel="Медианная стоимость, 100 тыс. долларов", ylabel="Число объектов",
       title="Распределение стоимости на train")
plt.show()
print("На верхней границе:", np.sum(y_train == 5.00001))

У правой границы заметен пик: 574 обучающих наблюдения имеют значение 5,00001. Стоимость в наборе данных ограничена сверху примерно 500 тыс. долларов. Прогнозировать будем именно записанное значение.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(train["MedInc"], y_train, s=8, alpha=0.2, color="steelblue")
ax.set(xlabel="Медианный доход, 10 тыс. долларов", ylabel="Стоимость, 100 тыс. долларов",
       title="Доход и стоимость жилья")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
points = ax.scatter(train["Longitude"], train["Latitude"], c=y_train,
                    s=8, alpha=0.65, cmap="viridis", vmin=0, vmax=5)
fig.colorbar(points, ax=ax, label="Стоимость, 100 тыс. долларов")
ax.set(xlabel="Долгота", ylabel="Широта", title="Географическое распределение стоимости")
plt.show()

Доход связан со стоимостью жилья, но территории с одинаковым доходом могут сильно различаться по ценам. На втором рисунке видны дорогие районы побережья и более дешёвые территории в глубине штата. Модель должна учитывать и доход, и местоположение, и их сочетания.

Основная метрика — RMSE:

$$\operatorname{RMSE}=\sqrt{\frac1n\sum_{i=1}^n(y_i-\hat y_i)^2}.$$

Например, RMSE 0,5 соответствует 50 тыс. долларов. Крупные ошибки особенно сильно влияют на метрику.

## 2. Как устроен случайный лес

Для каждого дерева случайный лес формирует бутстреп-выборку строк. Внутри дерева при поиске разбиения случайно выбираются признаки-кандидаты. Регрессионный лес усредняет прогнозы всех $B$ деревьев:

$$\hat f_{\mathrm{RF}}(x)=\frac1B\sum_{b=1}^B T_b(x).$$

Мы уже умеем строить дерево. Используем `DecisionTreeRegressor` для поиска порогов и формирования листьев, а самостоятельно реализуем создание обучающих наборов и объединение прогнозов.

### Бутстреп-индексы

`np.random.default_rng(SEED)` создаёт генератор случайных чисел. При одинаковом seed и одинаковой последовательности вызовов получаем одинаковые результаты. Каждый новый вызов генератора продолжает эту последовательность.

Для массива из пяти ответов `rng.choice(5, size=5, replace=True)` выбирает пять индексов из `0, 1, 2, 3, 4`. Первый аргумент задаёт множество индексов, `size` — число извлечений, `replace=True` — выбор с возвращением. Генератору не передаются значения ответов. Извлекаем их сами с помощью `small_y[indices]`.

In [ ]:
rng = np.random.default_rng(SEED)
small_y = np.array([2, 4, 7, 9, 13])

n = len(small_y)
indices = rng.choice(n, size=n, replace=True)
bootstrap_y = small_y[indices]

pd.DataFrame({"Индекс исходной строки": indices,
              "Извлечённый ответ": bootstrap_y})

Индексы 3 и 2 встретились по два раза, а индексы 1 и 4 отсутствуют. Всего осталось пять извлечений. В реальной таблице по одним индексам извлекаем строки `X[indices]` и ответы `y[indices]`, сохраняя соответствие между ними. Сохраним индексы каждого бутстрепа для OOB.

### Учебный класс `MyRandomForestRegressor`

В `__init__` сохраняем настройки будущего леса:

| Параметр | Что регулирует |
|---|---|
| `n_estimators` | Число деревьев в ансамбле |
| `max_depth` | Максимальную глубину каждого дерева (`None` — без ограничения) |
| `max_features` | Число кандидатов при поиске разбиения в узле: целое число или доля признаков; `1.0` — все признаки |
| `min_samples_leaf` | Минимальное число наблюдений в листе |
| `random_state` | Начальное состояние генератора для воспроизводимости |

В `fit` повторяем три действия: выбираем строки, обучаем дерево на этих строках, сохраняем модель и индексы. В `predict` сначала собираем прогнозы всех деревьев, затем усредняем их. Суффикс `_` у `estimators_` и `bootstrap_indices_` обозначает атрибуты, появляющиеся после обучения.

В sklearn `max_features` регулирует размер случайного набора кандидатов; если допустимого разбиения в нём нет, поиск может проверить дополнительные признаки.

In [ ]:
class MyRandomForestRegressor:
    """Учебный случайный лес для регрессии на числовых данных."""

    def __init__(self, n_estimators=100, max_depth=None,
                 max_features=1.0, min_samples_leaf=1, random_state=None):
        """Сохраняем число деревьев, их настройки и seed."""
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        """Обучаем деревья на бутстрепах и сохраняем их индексы."""
        X = np.asarray(X)
        y = np.asarray(y)
        if self.n_estimators < 1:
            raise ValueError("Число деревьев должно быть положительным")

        # При новом fit начинаем обучение заново.
        rng = np.random.default_rng(self.random_state)
        self.estimators_ = []
        self.bootstrap_indices_ = []
        n = len(y)

        for _ in range(self.n_estimators):
            # n случайных индексов; одну строку можно выбрать несколько раз.
            indices = rng.choice(n, size=n, replace=True)
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]

            # Для каждого дерева генерируем отдельный seed.
            tree_seed = int(rng.integers(0, 2**31 - 1))
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                max_features=self.max_features,  # кандидаты в каждом узле
                min_samples_leaf=self.min_samples_leaf,
                random_state=tree_seed,
            )
            tree.fit(X_bootstrap, y_bootstrap)

            # Позднее используем деревья для прогноза, индексы — для OOB.
            self.estimators_.append(tree)
            self.bootstrap_indices_.append(indices)
        return self

    def predict(self, X):
        """Возвращаем средний прогноз деревьев для каждого объекта."""
        X = np.asarray(X)
        predictions = []
        for tree in self.estimators_:
            predictions.append(tree.predict(X))

        # Строки — деревья, столбцы — объекты.
        predictions = np.array(predictions)
        return predictions.mean(axis=0)

`rng.integers(0, 2**31 - 1)` генерирует целое число для `random_state` дерева (верхняя граница исключена). Поэтому деревья могут по-разному выбирать признаки, а весь лес остаётся воспроизводимым. Базовое дерево sklearn выполняет рекурсивный поиск разбиений, который мы изучали на неделе 8. `return self` позволяет пользоваться привычным интерфейсом `fit`/`predict`.

### Проверка на маленьком лесе

Обучим пять неглубоких деревьев. Ограничение глубины позволит рассмотреть их структуру. Для сравнения качества дальше будем использовать более гибкие деревья.

In [ ]:
my_forest = MyRandomForestRegressor(n_estimators=5, max_depth=3,
                                   max_features=2, random_state=SEED)
my_forest.fit(X_train, y_train)
print("Число деревьев:", len(my_forest.estimators_))
print("Размер бутстрепа:", len(my_forest.bootstrap_indices_[0]))
print("Доля уникальных строк:", len(np.unique(my_forest.bootstrap_indices_[0])) / len(X_train))

В первом бутстрепе уникальны около 63,4% исходных строк — близко к теоретическим 63,2%. Остальные извлечения повторяют уже выбранные строки.

In [ ]:
X_example = X_val[:4]
example_predictions = np.array([tree.predict(X_example)
                                for tree in my_forest.estimators_])
pd.DataFrame(example_predictions,
             index=[f"Дерево {b+1}" for b in range(5)],
             columns=[f"Объект {i+1}" for i in range(4)]).round(3)

В каждой строке — четыре прогноза одного дерева. В каждом столбце — пять прогнозов одного объекта. `axis=0` означает, что усредняем по строкам и получаем по одному числу на объект. Например, среднее пяти прогнозов для первого объекта — около 2,166 (216,6 тыс. долларов).

In [ ]:
print("Форма:", example_predictions.shape)
print("Среднее:", example_predictions.mean(axis=0))
print("predict:", my_forest.predict(X_example))
print("Совпадают:", np.allclose(example_predictions.mean(axis=0), my_forest.predict(X_example)))

Рассмотрим верхние уровни двух деревьев. `plot_tree(max_depth=1)` ограничивает глубину рисунка; обученные деревья сохраняют свою исходную структуру.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for b, ax in enumerate(axes):
    plot_tree(my_forest.estimators_[b], feature_names=feature_names,
              max_depth=1, filled=True, rounded=True, fontsize=9, ax=ax)
    ax.set_title(f"Дерево {b+1}: верхние уровни")
plt.show()

Оба дерева начинают с `AveRooms`, но выбирают разные пороги (6,406 и 6,504). На следующем уровне набор разбиений уже отличается. Состав бутстрепа и случайный выбор кандидатов позволяют получить разные деревья.

### Три способа построить ансамбль в sklearn

Начнём с обычного бэггинга. `BaggingRegressor` принимает базовый алгоритм (`estimator`), число моделей (`n_estimators`) и способ выбора строк (`bootstrap=True` — с возвращением).

In [ ]:
bagging = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=5,
    bootstrap=True,
    random_state=SEED,
)
bagging

### Бэггинг деревьев со случайными разбиениями

Если передать в `estimator` дерево с `max_features=2`, то каждое дерево будет искать разбиение среди двух случайных кандидатов. Так можно собрать механизм случайного леса из двух библиотечных классов. У самого `BaggingRegressor` тоже есть `max_features`, но он выбирает столбцы один раз для всей базовой модели. Здесь оставляем этот внешний параметр равным `1.0` (все столбцы).

In [ ]:
bagging_with_random_splits = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_features=2),
    n_estimators=5,
    max_features=1.0,
    bootstrap=True,
    random_state=SEED,
)
bagging_with_random_splits

### RandomForestRegressor

В специализированном классе параметры деревьев передаются напрямую. Обучим маленький лес: пять деревьев, не более трёх разбиений вдоль пути от корня, два признака-кандидата в узле, бутстреп-выборки строк. `n_jobs` задаёт число параллельных задач при обучении леса: `1` — последовательное обучение деревьев, `-1` — по числу доступных логических CPU.

In [ ]:
rf_small = RandomForestRegressor(
    n_estimators=5,
    max_depth=3,
    max_features=2,
    bootstrap=True,
    random_state=SEED,
    n_jobs=1,
)
rf_small.fit(X_train, y_train)

У библиотечного леса обученные деревья хранятся в `estimators_`. Проверим глубину, количество листьев и признаки в корне. Затем воспроизведём среднее прогнозов.

In [ ]:
pd.DataFrame({
    "Дерево": np.arange(1, 6),
    "Признак в корне": [feature_names[t.tree_.feature[0]] for t in rf_small.estimators_],
    "Глубина": [t.get_depth() for t in rf_small.estimators_],
    "Число листьев": [t.get_n_leaves() for t in rf_small.estimators_],
})

In [ ]:
library_predictions = np.array([t.predict(X_example) for t in rf_small.estimators_])
print("Среднее:", library_predictions.mean(axis=0))
print("Прогноз RF:", rf_small.predict(X_example))
print("Совпадают:", np.allclose(library_predictions.mean(axis=0), rf_small.predict(X_example)))

Среднее прогнозов снова совпало с `predict`. Это тот же принцип, что в нашем классе. Точные прогнозы двух реализаций могут различаться из-за разных бутстрепов и случайных разбиений.

## 3. Что выигрываем от перехода к лесу?

До сих пор обучали маленькие леса для проверки алгоритма. Теперь обучим отдельные модели для сравнения качества на общих train и validation. Возьмём несколько разумных настроек каждого семейства. Вспомогательная функция обучает кандидатов и сортирует их по RMSE.

In [ ]:
def evaluate_candidates(candidates):
    rows = []
    for name, model in candidates:
        start = perf_counter()
        model.fit(X_train, y_train)
        seconds = perf_counter() - start
        prediction = model.predict(X_val)
        rows.append({"configuration": name, "RMSE": root_mean_squared_error(y_val, prediction),
                     "fit_seconds": seconds})
    return pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

### Знакомые модели

Начнём с постоянного прогноза (`DummyRegressor`), затем Ridge с разной силой регуляризации `alpha` и kNN с разным числом соседей и весами (`uniform` — равные, `distance` — по расстоянию). Масштабирование выполняется внутри pipeline.

In [ ]:
dummy = DummyRegressor(strategy="mean")
dummy_results = evaluate_candidates([("mean", dummy)])
dummy_results

In [ ]:
ridge_candidates = []
for alpha in [0.01, 1, 100]:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    ridge_candidates.append((f"alpha={alpha}", model))
ridge_results = evaluate_candidates(ridge_candidates)
ridge_results

In [ ]:
knn_candidates = []
for k in [3, 10, 30]:
    for weights in ["uniform", "distance"]:
        model = make_pipeline(StandardScaler(),
                              KNeighborsRegressor(n_neighbors=k, weights=weights))
        knn_candidates.append((f"k={k}, weights={weights}", model))
knn_results = evaluate_candidates(knn_candidates)
knn_results

Константа даёт RMSE 1,159, Ridge — 0,707, kNN — 0,636. Нелинейная локальная модель здесь точнее линейной. Теперь попробуем дерево: как на неделе 8, подберём `max_depth` и `min_samples_leaf`.

In [ ]:
tree_candidates = []
for depth in [4, 8, 16, None]:
    for leaf in [1, 5]:
        model = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf,
                                      random_state=SEED)
        tree_candidates.append((f"depth={depth}, leaf={leaf}", model))
tree_results = evaluate_candidates(tree_candidates)
tree_results

Лучшее дерево в сетке имеет глубину 8 и минимум 5 объектов в листе (RMSE 0,647). Глубокое дерево без ограничений хуже: 0,751. Может ли усреднение помочь использовать такие гибкие деревья?

### Bagging

Для сравнения используем один класс `RandomForestRegressor`: с `max_features=1.0` доступны все признаки, получаем обычный бэггинг. Обучим 80 деревьев, проверим два значения `min_samples_leaf`.

In [ ]:
bag_candidates = []
for leaf in [1, 3]:
    model = RandomForestRegressor(n_estimators=80, max_features=1.0,
                                  min_samples_leaf=leaf, bootstrap=True,
                                  random_state=SEED, n_jobs=1)
    bag_candidates.append((f"leaf={leaf}", model))
bag_results = evaluate_candidates(bag_candidates)
bag_results

Усреднение глубоких деревьев даёт заметно меньшую ошибку: 0,505 против 0,647 у настроенного одиночного дерева.

### Random Forest

Оставим 80 деревьев и те же варианты размера листа. Теперь ограничим число кандидатов в узле: 2, 4 или 6. Посмотрим, даёт ли дополнительная рандомизация выигрыш.

In [ ]:
rf_candidates = []
for m in [2, 4, 6]:
    for leaf in [1, 3]:
        model = RandomForestRegressor(n_estimators=80, max_features=m,
                                      min_samples_leaf=leaf, bootstrap=True,
                                      random_state=SEED, n_jobs=1)
        rf_candidates.append((f"features={m}, leaf={leaf}", model))
rf_results = evaluate_candidates(rf_candidates)
rf_results

In [ ]:
# Обученные кандидаты больше не нужны; оставляем таблицы результатов.
del bag_candidates, rf_candidates, tree_candidates, knn_candidates, ridge_candidates, model

### Итог сравнения

Возьмём лучшую настройку каждого семейства по validation.

In [ ]:
tables = {"Dummy": dummy_results, "Ridge": ridge_results,
          "kNN": knn_results, "Tree": tree_results,
          "Bagging": bag_results, "RF": rf_results}
rows = []
for name, table in tables.items():
    best = table.iloc[0]  # таблица уже отсортирована по RMSE
    rows.append({"model": name, "configuration": best["configuration"],
                 "RMSE": best["RMSE"]})
results = pd.DataFrame(rows)
results

In [ ]:
fig, ax = plt.subplots()
ax.bar(results["model"], results["RMSE"], color="steelblue")
for i, v in enumerate(results["RMSE"]):
    ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)
ax.set(ylabel="Validation RMSE", title="Лучшие конфигурации семейств моделей",
       ylim=(0, results["RMSE"].max() * 1.12))
plt.show()

Основной выигрыш даёт переход от дерева к бэггингу: RMSE падает с 64,7 до 50,5 тыс. долларов. RF улучшает результат ещё примерно на 900 долларов. На следующем занятии добавим к сравнению бустинг. Пока все настройки и оценки получены на validation, test оставляем закрытым.

## 4. Сколько деревьев нужно?

В сравнении мы использовали 80 деревьев. Что будет, если обучить больше? Построим четыре новых леса по 160 глубоких деревьев. Они различаются **только `max_features`**: 1, 2, 4, 8. Последний — бэггинг. Размер листа равен 1, глубина не ограничена.

Эти же четыре леса пригодятся в следующем разделе. Сохраняем для каждого леса прогнозы всех деревьев на validation и число узлов; после этого сами модели можно удалить.

In [ ]:
B = 160
validation_predictions_by_m = {}
fit_seconds_by_m = {}
node_counts_by_m = {}
for m in [1, 2, 4, 8]:
    forest = RandomForestRegressor(n_estimators=B, max_features=m,
                                   bootstrap=True, random_state=SEED, n_jobs=1)
    start = perf_counter()
    forest.fit(X_train, y_train)
    fit_seconds_by_m[m] = perf_counter() - start

    # Сохраняем прогнозы и размеры деревьев. Сам лес больше не понадобится.
    predictions = []
    sizes_of_trees = []
    for tree in forest.estimators_:
        predictions.append(tree.predict(X_val))
        sizes_of_trees.append(tree.tree_.node_count)
    validation_predictions_by_m[m] = np.array(predictions)
    node_counts_by_m[m] = np.array(sizes_of_trees)
    if m == 2:
        example_tree_info = {
            "Глубина": forest.estimators_[0].get_depth(),
            "Листья": forest.estimators_[0].get_n_leaves(),
            "Узлы": forest.estimators_[0].tree_.node_count,
            "Различные строки бутстрепа": len(np.unique(forest.estimators_samples_[0])),
        }
    del forest, tree
print("Форма массива прогнозов:", validation_predictions_by_m[2].shape)

Например, `validation_predictions_by_m[2]` хранит прогнозы леса с двумя кандидатами: 160 строк (деревья) и 4 128 столбцов (территории validation).

Чтобы узнать качество ансамбля из 20 деревьев, достаточно взять первые 20 строк и усреднить их (`[:20].mean(axis=0)`). Затем считаем RMSE. Повторим для разных размеров ансамбля; уже включённые деревья при этом не меняются.

In [ ]:
sizes = [1, 2, 5, 10, 20, 40, 80, 160]
curve_rows = []
for m, tree_predictions in validation_predictions_by_m.items():
    for b in sizes:
        prediction = tree_predictions[:b].mean(axis=0)
        curve_rows.append({"max_features": m, "n_estimators": b,
                           "RMSE": root_mean_squared_error(y_val, prediction)})
curves = pd.DataFrame(curve_rows)

In [ ]:
fig, ax = plt.subplots()
for m, group in curves.groupby("max_features"):
    label = "Bagging (8)" if m == 8 else f"RF ({m})"
    ax.plot(group["n_estimators"], group["RMSE"], marker="o", label=label)
ax.set(xlabel="Число деревьев", ylabel="Validation RMSE", title="Качество среднего прогноза")
ax.set_xscale("log", base=2)
ax.set_xticks(sizes, labels=sizes)
ax.legend(title="Число кандидатов")
ax.grid(alpha=0.2)
plt.show()

По горизонтали — число деревьев (логарифмическая шкала). Для леса с двумя кандидатами RMSE падает с 0,781 у первого дерева до 0,536 при десяти деревьях. Затем кривая выравнивается: 0,508 при 40 и 0,493 при 160. Большую часть выигрыша даёт относительно небольшой ансамбль.

Вспомним формулу с лекции:

$$\operatorname{Var}(\bar T_B)=\tau^2\left(\rho+\frac{1-\rho}{B}\right).$$

При фиксированных $\rho$ и $\tau^2$ второе слагаемое быстро уменьшается. На рисунке показана проверочная ошибка, а не дисперсия прогноза; небольшие подъёмы кривой возможны.

### Цена большого леса

В лесу с двумя кандидатами переход от 80 к 160 деревьям улучшает RMSE примерно на 600 долларов. Насколько при этом вырастет модель? Посмотрим, сколько узлов содержит одно глубокое дерево. `get_depth()` возвращает глубину, `get_n_leaves()` — число листьев, `tree_.node_count` — число всех узлов.

In [ ]:
pd.Series(example_tree_info, name="Первое дерево")

У первого дерева глубина 37, но узлов больше 15 тысяч. Глубина — длина самого длинного пути от корня к листу. Число узлов зависит и от того, сколько ветвей выросло.

В бутстрепе около 7 800 различных строк; глубокое дерево может почти каждую из них выделить в отдельный лист. Отсюда и такой размер. Посчитаем суммарное число узлов при 20, 80 и 160 деревьях.

In [ ]:
node_counts = node_counts_by_m[2]
print("Среднее число узлов одного дерева:", round(node_counts.mean()))
rows = []
for b in [20, 80, 160]:
    total_nodes = node_counts[:b].sum()
    rows.append({"Число деревьев": b, "Всего узлов": total_nodes})
pd.DataFrame(rows)

При 80 деревьях получаем около 1,22 млн узлов, при 160 — около 2,44 млн. Размер структуры почти удваивается ради небольшого выигрыша в RMSE. Поэтому число деревьев выбирают с учётом времени, памяти и нужной точности. Ограничение глубины и увеличение `min_samples_leaf` тоже позволяют уменьшить размер модели.

## 5. Сколько признаков давать дереву?

На предыдущем рисунке вариант со всеми восемью признаками проиграл лесу с двумя кандидатами. На первый взгляд это странно: чем больше признаков доступно дереву, тем больше возможностей найти хорошее разбиение. Выясним, что происходит.

Продолжаем работать **с теми же четырьмя лесами из раздела 4** (по 160 деревьев, `max_features` = 1, 2, 4, 8). Новые модели не обучаем.

### 5.1. Насколько похожи деревья?

Сравним два крайних случая: лес с одним кандидатом и бэггинг со всеми восемью. Возьмём первые два дерева каждого леса. На диаграммах каждая точка — территория validation; координаты — прогнозы двух деревьев. Пунктир соответствует равным прогнозам. Коэффициент Пирсона измеряет линейную связь между прогнозами. Здесь меняются проверочные объекты, а деревья уже обучены; на лекции мы изучали корреляцию при повторном обучении в фиксированной точке.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout="constrained")
for m, ax in zip([1, 8], axes):
    predictions = validation_predictions_by_m[m]
    first_tree = predictions[0]
    second_tree = predictions[1]
    # np.corrcoef возвращает матрицу 2 × 2; [0, 1] — корреляция пары.
    r = np.corrcoef(first_tree, second_tree)[0, 1]
    ax.scatter(first_tree, second_tree, s=8, alpha=0.18)
    ax.plot([0, 5.1], [0, 5.1], "--", color="gray", linewidth=1)
    ax.set(xlim=(0, 5.1), ylim=(0, 5.1), aspect="equal",
           xlabel="Прогноз дерева 1", ylabel="Прогноз дерева 2",
           title=f"{m} из 8 кандидатов; r = {r:.3f}")
plt.show()

Справа облако заметно теснее вытянуто вдоль диагонали: прогнозы двух деревьев лучше согласуются (корреляция 0,769 против 0,596 слева). Но мы посмотрели только на одну пару в каждом лесу. Для сравнения лесов целиком посмотрим на все пары. В каждом массиве прогнозов строки соответствуют деревьям, столбцы — территориям. `np.corrcoef(predictions)` строит матрицу 160 × 160: элемент $(b,c)$ — корреляция прогнозов деревьев $b$ и $c$ по всем территориям validation.

На диагонали единицы, каждая пара вне диагонали встречается дважды. `np.triu_indices_from(matrix, k=1)` выбирает клетки строго выше диагонали; среднее по ним учитывает каждую пару один раз. Постоянные прогнозы исключаем: для них корреляция не определена.

In [ ]:
def prediction_correlations(predictions):
    """Матрица корреляций непостоянных прогнозов и среднее по парам."""
    nonconstant = predictions.std(axis=1) > 0
    if nonconstant.sum() < 2:
        return np.eye(nonconstant.sum()), np.nan
    matrix = np.corrcoef(predictions[nonconstant])
    # k=1: индексы строго выше главной диагонали.
    upper = np.triu_indices_from(matrix, k=1)
    return matrix, matrix[upper].mean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for m, ax in zip([1, 8], axes):
    matrix, mean_corr = prediction_correlations(validation_predictions_by_m[m])
    image = ax.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm", interpolation="nearest")
    ax.set(title=f"max_features={m}; среднее={mean_corr:.3f}", xlabel="Дерево", ylabel="Дерево")
fig.colorbar(image, ax=axes, label="Корреляция прогнозов")
plt.show()

Правая матрица темнее: средняя корреляция выросла с 0,600 до 0,763. При восьми кандидатах разные деревья могут чаще выбирать одни и те же сильные признаки (например, доход или широту). Ограничивая число кандидатов, получаем более разнообразные деревья — именно ради этого вводили дополнительную рандомизацию.

Как дополнительная рандомизация сказывается на точности отдельных деревьев и всего леса?

### 5.2. Качество дерева и качество ансамбля

Рассмотрим три условных объекта:

| | Объект 1 | Объект 2 | Объект 3 | RMSE |
|---|---:|---:|---:|---:|
| Фактический ответ | 2 | 3 | 4 | — |
| Дерево 1 | 3 | 4 | 5 | 1 |
| Дерево 2 | 1 | 2 | 3 | 1 |
| Средний прогноз | 2 | 3 | 4 | 0 |

У обоих деревьев RMSE равна 1, у ансамбля — 0. Ошибки компенсировались при усреднении прогнозов.

Для каждого леса посчитаем две величины:

- **Средняя RMSE дерева:** считаем RMSE каждого из 160 деревьев на validation, затем усредняем 160 ошибок.
- **RMSE леса:** сначала усредняем 160 прогнозов для каждой территории, затем считаем RMSE среднего прогноза.

Для примера возьмём лес с двумя кандидатами из раздела 4.

In [ ]:
predictions_m2 = validation_predictions_by_m[2]
# Сначала ошибки отдельных деревьев, затем среднее ошибок.
tree_errors = []
for tree_prediction in predictions_m2:
    tree_errors.append(root_mean_squared_error(y_val, tree_prediction))
mean_tree_rmse = np.mean(tree_errors)

# Сначала средний прогноз, затем его ошибка.
forest_prediction = predictions_m2.mean(axis=0)
forest_rmse = root_mean_squared_error(y_val, forest_prediction)

print(f"Средняя RMSE дерева: {mean_tree_rmse:.3f}")
print(f"RMSE леса: {forest_rmse:.3f}")

У отдельного дерева в среднем RMSE 0,799; после усреднения прогнозов 160 деревьев получаем 0,493. Это выигрыш от ансамблирования. Повторим тот же расчёт для всех четырёх лесов и добавим среднюю корреляцию прогнозов.

In [ ]:
feature_rows = []
for m, tree_predictions in validation_predictions_by_m.items():
    # Качество отдельных деревьев.
    tree_errors = []
    for tree_prediction in tree_predictions:
        tree_errors.append(root_mean_squared_error(y_val, tree_prediction))
    mean_tree_rmse = np.mean(tree_errors)

    # Сходство деревьев.
    _, mean_corr = prediction_correlations(tree_predictions)

    # Качество всего леса.
    forest_prediction = tree_predictions.mean(axis=0)
    forest_rmse = root_mean_squared_error(y_val, forest_prediction)

    feature_rows.append({"max_features": m, "mean_tree_RMSE": mean_tree_rmse,
                         "mean_correlation": mean_corr, "forest_RMSE": forest_rmse})
feature_results = pd.DataFrame(feature_rows)
feature_results.round(4)

### 5.3. Где находится удачный компромисс?

Поставим рядом три графика. По горизонтали везде число признаков-кандидатов в узле; по вертикали — три характеристики из таблицы.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout="constrained")
for ax, col, label in zip(axes,
        ["mean_tree_RMSE", "mean_correlation", "forest_RMSE"],
        ["Средняя RMSE дерева", "Средняя корреляция", "RMSE леса"]):
    ax.plot(feature_results["max_features"], feature_results[col], marker="o")
    ax.set(xlabel="max_features", ylabel=label, xticks=[1, 2, 4, 8])
    ax.grid(alpha=0.2)
plt.show()

Слева: с ростом числа кандидатов отдельные деревья становятся точнее (RMSE снижается с 0,898 до 0,755). В середине: одновременно растёт сходство прогнозов. Справа: самый точный ансамбль среди четырёх вариантов — с двумя кандидатами (0,493), хотя его базовые деревья уступают деревьям бэггинга. Выигрыш от усреднения перекрывает эту разницу. При одном кандидате базовые деревья слишком неточны.

Так проявляется компромисс, ради которого вводим случайный выбор признаков. На других данных лучшим может оказаться и бэггинг. Поэтому `max_features` подбирают по качеству **ансамбля**.

## 6. Out-of-bag: как проверить лес без отдельной validation?

Для обучающего объекта соберём прогноз из деревьев, которые не получили его в своём бутстрепе (если такие деревья есть). Воспользуемся этим, чтобы оценить качество и подобрать настройки леса.

### 6.1. OOB-прогноз вручную

Вернёмся к `my_forest` из раздела 2: пяти неглубоким деревьям с сохранёнными бутстреп-индексами. Выберем объект с индексом 1 из train. Посчитаем, сколько раз он появился в бутстрепе каждого дерева, и запишем прогноз.

In [ ]:
i = 1
rows = []
for b, tree in enumerate(my_forest.estimators_):
    indices = my_forest.bootstrap_indices_[b]
    count = np.sum(indices == i)  # сколько раз индекс встретился в бутстрепе
    prediction = tree.predict(X_train[i:i+1])[0]
    rows.append({"Дерево": b + 1, "Вхождений в бутстреп": count,
                 "Прогноз": prediction})
oob_example = pd.DataFrame(rows)
oob_example

Деревья 1 и 2 не использовали объект: у них ноль в столбце «Вхождений в бутстреп». Только их прогнозы возьмём для OOB. Булева маска в pandas позволяет оставить именно эти две строки.

In [ ]:
eligible = oob_example[oob_example["Вхождений в бутстреп"] == 0]
if eligible.empty:
    print("Для этого объекта OOB-прогноз не определён")
else:
    print("Подходящие деревья:", eligible["Дерево"].tolist())
    print("OOB-прогноз:", round(eligible["Прогноз"].mean(), 3))
    print("Фактический ответ:", y_train[i])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for _, row in oob_example.iterrows():
    usable = row["Вхождений в бутстреп"] == 0
    ax.scatter(row["Дерево"], row["Прогноз"], s=95,
               color="seagreen" if usable else "gray", marker="o" if usable else "x")
if not eligible.empty:
    ax.axhline(eligible["Прогноз"].mean(), color="seagreen", linestyle="--", label="OOB-прогноз")
ax.axhline(y_train[i], color="black", linestyle=":", label="Фактический ответ")
ax.set(xlabel="Дерево", ylabel="Прогноз", xticks=range(1, 6),
       title="Зелёные точки: деревья, не обучавшиеся на объекте")
ax.legend()
plt.show()

По двум зелёным точкам получаем прогноз 1,572 (157,2 тыс. долларов), фактическая стоимость — 118,8 тыс. Ошибка составляет около 38,4 тыс. долларов. Остальные три дерева в расчёт не входят.

Для маленького леса возможен и другой случай: если объект попал во все пять бутстрепов, его OOB-прогноз не определён. С увеличением числа деревьев такие ситуации становятся редкими.

### 6.2. OOB в sklearn

Обучим отдельный лес `rf_oob` из 80 деревьев с четырьмя кандидатами. Параметр `oob_score=True` включает вычисление OOB-прогнозов; нужен `bootstrap=True`. После `fit` атрибут `oob_prediction_` содержит прогнозы для обучающих объектов, а `oob_score_` — $R^2$ по умолчанию для регрессии.

In [ ]:
rf_oob = RandomForestRegressor(n_estimators=80, max_features=4,
                               bootstrap=True, oob_score=True,
                               random_state=SEED, n_jobs=1)
rf_oob.fit(X_train, y_train)

Сколько деревьев реально участвует в OOB-прогнозе? На лекции получили вероятность около $1/e$ для одного дерева. Для 80 деревьев ожидаем примерно 29.

`estimators_samples_` хранит бутстреп-индексы каждого дерева. В функции ниже отмечаем использованные строки булевой маской; `~used` отмечает оставшиеся. Складываем эти маски по деревьям, чтобы получить число OOB-деревьев для каждого объекта.

In [ ]:
def count_oob_trees(model, n_objects):
    """Число деревьев, не использовавших каждый обучающий объект."""
    counts = np.zeros(n_objects, dtype=int)
    for indices in model.estimators_samples_:
        used = np.zeros(n_objects, dtype=bool)
        used[indices] = True
        counts += ~used
    return counts

counts = count_oob_trees(rf_oob, len(y_train))
print("Минимум OOB-деревьев:", counts.min())
print("Строк без OOB-прогноза:", np.sum(counts == 0))
print("Средняя доля OOB-деревьев:", round(counts.mean() / rf_oob.n_estimators, 3))

In [ ]:
fig, ax = plt.subplots()
ax.hist(counts, bins=np.arange(counts.min()-0.5, counts.max()+1.5), edgecolor="white")
ax.axvline(80 / np.e, color="black", linestyle="--", label="80 / e")
ax.set(xlabel="Число OOB-деревьев для объекта", ylabel="Число объектов",
       title="Сколько деревьев участвует в OOB-прогнозах?")
ax.legend()
plt.show()

Средняя доля подходящих деревьев 0,368 — почти точное совпадение с теоретическим результатом. По горизонтали гистограммы — число OOB-деревьев, по вертикали — число обучающих объектов. Пунктир отмечает ожидаемое число $80/e$. Даже у объекта с самым маленьким числом подходящих деревьев их 12; OOB-прогноз есть для каждой строки. Теперь посчитаем общую ошибку.

`counts[i]` — число деревьев, подходящих для OOB-прогноза объекта с позицией `i`. Выражение `counts > 0` создаёт булеву маску: например, для `[2, 0, 3]` получим `[True, False, True]`. Индексация `y_train[covered]` и `rf_oob.oob_prediction_[covered]` оставляет одни и те же объекты, для которых есть OOB-прогноз. В нашем лесу все объекты покрыты; маска пригодится, если число деревьев будет небольшим.

In [ ]:
covered = counts > 0
print("OOB R²:", rf_oob.oob_score_)
print("OOB RMSE:", root_mean_squared_error(y_train[covered], rf_oob.oob_prediction_[covered]))
print("Validation RMSE:", root_mean_squared_error(y_val, rf_oob.predict(X_val)))

OOB RMSE — 51,4 тыс. долларов, обычная validation RMSE — 49,6 тыс. Разница небольшая: оценки сопоставимого масштаба. OOB-прогноз использует в среднем 29 деревьев вместо 80, а наборы проверочных объектов различаются; обе причины могут влиять на оценки. В этой задаче мы получили полезную оценку качества прямо при обучении.

### 6.3. Подбираем параметры по OOB

Для выбора `max_features` и `min_samples_leaf` переберём одну и ту же сетку параметров двумя способами: оценим каждый лес по OOB и по кросс-валидации. OOB удобен, когда подбираем параметры только случайного леса на независимых наблюдениях. CV позволяет использовать единый протокол для разных алгоритмов и, при необходимости, специальные разбиения данных. В обоих случаях здесь работаем только на train.

`ParameterGrid` из sklearn превращает словарь списков в перебор всех шести сочетаний. `**params` передаёт очередное сочетание как именованные аргументы конструктора леса.

In [ ]:
param_grid = {
    "max_features": [2, 4, 8],
    "min_samples_leaf": [1, 5],
}
list(ParameterGrid(param_grid))

In [ ]:
oob_rows = []
start = perf_counter()
for params in ParameterGrid(param_grid):
    model = RandomForestRegressor(
        n_estimators=80, bootstrap=True, oob_score=True,
        random_state=SEED, n_jobs=1, **params,
    )
    model.fit(X_train, y_train)
    counts = count_oob_trees(model, len(y_train))
    if np.any(counts == 0):
        raise ValueError("Есть объекты без OOB-прогноза")
    error = root_mean_squared_error(y_train, model.oob_prediction_)
    oob_rows.append({**params, "OOB_RMSE": error})
    del model

oob_seconds = perf_counter() - start
oob_results = pd.DataFrame(oob_rows).sort_values("OOB_RMSE")
oob_results.round(4)

OOB выбирает четыре кандидата и один объект в листе (RMSE 0,514). Теперь посмотрим, что выберет трёхфолдовая CV.

В `GridSearchCV` используем ту же сетку. `scoring="neg_root_mean_squared_error"` — RMSE с минусом, поскольку sklearn максимизирует score. `refit=False` отключает финальное переобучение победителя: сейчас сравниваем именно стоимость поиска. `KFold` задаёт три перемешанных фолда; `n_jobs=1` сохраняет один режим вычислений в обоих экспериментах.

In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=SEED)
search = GridSearchCV(
    estimator=RandomForestRegressor(n_estimators=80, bootstrap=True,
                                    random_state=SEED, n_jobs=1),
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    refit=False,
    n_jobs=1,
)
start = perf_counter()
search.fit(X_train, y_train)
cv_seconds = perf_counter() - start
print("Лучшие параметры:", search.best_params_)
print("Средняя CV RMSE:", -search.best_score_)

CV тоже выбирает четыре кандидата и один объект в листе. Соберём результаты из `cv_results_`: `mean_test_score` содержит средний score по фолдам; меняем знак и получаем RMSE. Сопоставим оценки по каждой конфигурации.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_["params"])
cv_results["CV_RMSE"] = -search.cv_results_["mean_test_score"]
comparison = oob_results.merge(cv_results, on=["max_features", "min_samples_leaf"])
comparison = comparison.sort_values(["max_features", "min_samples_leaf"])
comparison.round(4)

Обе процедуры выбрали один вариант. В CV лес с двумя кандидатами практически не уступает лесу с четырьмя. Для нашего выбора OOB оказался достаточным. А что с затратами на поиск?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
seconds = [oob_seconds, cv_seconds]
bars = ax.bar(["OOB: 6 лесов", "3-fold CV: 18 лесов"], seconds,
              color=["seagreen", "steelblue"])
ax.bar_label(bars, fmt="%.1f с", padding=4)
ax.set(ylabel="Секунды", title="Время перебора шести конфигураций",
       ylim=(0, max(seconds) * 1.22))
plt.show()

В этом запуске подбор по OOB занял 17,4 секунды, по CV — 31,8 секунды. Причина выигрыша проста: OOB требует одного обучения на конфигурацию, CV — трёх. В нашей сетке это шесть обучений вместо восемнадцати. Ускорение не обязано быть трёхкратным: в CV лес обучается на меньшей выборке, а OOB-прогнозы тоже нужно вычислять. При подборе настроек по OOB итоговый test всё равно нужен. Для временных или сгруппированных данных обычный построчный OOB может не соответствовать задаче прогноза.

## Итоги

На California Housing основное улучшение качества приносит усреднение деревьев; случайный выбор признаков дополнительно помогает в этом эксперименте. Увеличивать число деревьев полезно до тех пор, пока выигрыш оправдывает вычислительные затраты. OOB позволяет оценить качество и подбирать гиперпараметры, обучая меньше моделей, чем при CV.

На следующем семинаре добавим градиентный бустинг: деревья будут строиться последовательно, исправляя ошибки предыдущего ансамбля. Сравним два способа построения ансамблей на тех же данных.

**Документация:** [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html), [BaggingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingRegressor.html), [ParameterGrid](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterGrid.html), [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), [OOB error](https://scikit-learn.org/stable/auto_examples/ensemble/plot_ensemble_oob.html).